In [2]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

In [3]:
DATA_PATH =Path("../data/raw")
print(DATA_PATH)

..\data\raw


In [4]:
csv_files = list(DATA_PATH.glob("*.csv"))

csv_files

[WindowsPath('../data/raw/olist_customers_dataset.csv'),
 WindowsPath('../data/raw/olist_geolocation_dataset.csv'),
 WindowsPath('../data/raw/olist_orders_dataset.csv'),
 WindowsPath('../data/raw/olist_order_items_dataset.csv'),
 WindowsPath('../data/raw/olist_order_payments_dataset.csv'),
 WindowsPath('../data/raw/olist_order_reviews_dataset.csv'),
 WindowsPath('../data/raw/olist_products_dataset.csv'),
 WindowsPath('../data/raw/olist_sellers_dataset.csv'),
 WindowsPath('../data/raw/product_category_name_translation.csv')]

In [5]:
datasets={}

for file in csv_files:
    datasets[file.stem] = pd.read_csv(file)
print(f"Loaded {len(datasets)} datasets successfully.")


Loaded 9 datasets successfully.


In [6]:
datasets.keys()

dict_keys(['olist_customers_dataset', 'olist_geolocation_dataset', 'olist_orders_dataset', 'olist_order_items_dataset', 'olist_order_payments_dataset', 'olist_order_reviews_dataset', 'olist_products_dataset', 'olist_sellers_dataset', 'product_category_name_translation'])

In [7]:
def profile_dataset(df, dataset_name):

    print("=" * 60)
    print(f"Dataset: {dataset_name}")
    print("=" * 60)

    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")

    print("\nColumn Names:")
    print(df.columns.tolist())

    print("\nData Types:")
    print(df.dtypes)

    print("\nMissing Values:")
    print(df.isnull().sum())

    print("\nDuplicate Rows:")
    print(df.duplicated().sum())

    print("\nMemory Usage:")
    print(f"{df.memory_usage(deep=True).sum() / 1024:.2f} KB")

    print("\nFirst Five Rows:")
    display(df.head())

In [8]:
profile_dataset(
    datasets["olist_orders_dataset"],
    "Orders Dataset"
)

Dataset: Orders Dataset
Rows: 99441
Columns: 8

Column Names:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Data Types:
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

Missing Values:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Duplicate Rows:
0

Memory Usage:
60384.49 KB

First Five Rows:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


### Findings
- Dataset contains 99,441 orders.
- No duplicate rows detected.
- Timestamp columns are stored as object type.
- Date conversion will be required before time-series analysis.


In [9]:

def profile_summary(datasets):

    summary = []

    for dataset_name, df in datasets.items():

        total_cells = df.shape[0] * df.shape[1]

        missing_values = df.isnull().sum().sum()

        duplicate_rows = df.duplicated().sum()

        object_cols = df.select_dtypes(include="object").shape[1]

        numeric_cols = df.select_dtypes(include=["int64", "float64"]).shape[1]

        summary.append({

            "Dataset": dataset_name,

            "Rows": df.shape[0],

            "Columns": df.shape[1],

            "Missing Values": missing_values,

            "Missing %": round((missing_values / total_cells) * 100, 2),

            "Duplicate Rows": duplicate_rows,

            "Duplicate %": round((duplicate_rows / df.shape[0]) * 100, 2),

            "Object Columns": object_cols,

            "Numeric Columns": numeric_cols,

            "Memory (KB)": round(df.memory_usage(deep=True).sum() / 1024, 2)

        })

    summary_df = pd.DataFrame(summary)

    return summary_df

    

In [10]:
summary = profile_summary(datasets)

display(summary)

,Dataset,Rows,Columns,Missing Values,Missing %,Duplicate Rows,Duplicate %,Object Columns,Numeric Columns,Memory (KB)
0,olist_customers_dataset,99441,5,0,0.00,0,0.00,4,1,30332.01
1,olist_geolocation_dataset,1000163,5,0,0.00,261831,26.18,2,3,149592.55
2,olist_orders_dataset,99441,8,4908,0.62,0,0.00,8,0,60384.49
3,olist_order_items_dataset,112650,7,0,0.00,0,0.00,4,3,40373.71
4,olist_order_payments_dataset,103886,5,0,0.00,0,0.00,2,3,18242.14
5,olist_order_reviews_dataset,99224,7,145903,21.01,0,0.00,6,1,43772.64
6,olist_products_dataset,32951,9,2448,0.83,0,0.00,2,7,6957.78
7,olist_sellers_dataset,3095,4,0,0.00,0,0.00,3,1,674.78
8,product_category_name_translation,71,2,0,0.00,0,0.00,2,0,10.32


In [11]:
def missing_value_report(df):

    missing = pd.DataFrame({

        "Column": df.columns,

        "Missing Values": df.isnull().sum().values,

        "Missing %": ((df.isnull().sum() / len(df)) * 100).round(2).values,

        "Data Type": df.dtypes.values

    })

    missing = missing.sort_values(
        by="Missing Values",
        ascending=False
    )

    return missing

In [12]:
orders_missing = missing_value_report(
    datasets["olist_orders_dataset"]
)

display(orders_missing)

,Column,Missing Values,Missing %,Data Type
6,order_delivered_customer_date,2965,2.98,object
5,order_delivered_carrier_date,1783,1.79,object
4,order_approved_at,160,0.16,object
0,order_id,0,0.00,object
3,order_purchase_timestamp,0,0.00,object
2,order_status,0,0.00,object
1,customer_id,0,0.00,object
7,order_estimated_delivery_date,0,0.00,object
